# Image writing test

This file is to test the writing speed of different file format(e.g. PNG and HDF5).
The HDF5 format should be included metadata of image with:
1. Time information
2. shear rate
3. Exposure
4. Experiment description

In [4]:
from pathlib import Path
import time
import cv2
import h5py
import numpy as np


src_dir = Path("E:/202606_experiment/20260622_125956/2_60.0")
png_out = Path("./writing_test/output_png")
hdf_out = Path("./writing_test/output_hdf5")

png_out.mkdir(exist_ok=True)
hdf_out.mkdir(exist_ok=True)

image_paths = sorted(src_dir.glob("*.png"))[:800]
batch_size = 40

img0 = cv2.imread(str(image_paths[0]), cv2.IMREAD_GRAYSCALE)
h, w = img0.shape
dtype = img0.dtype
n = len(image_paths)

print(f"Images: {n}")
print(f"Shape: {h} x {w}")
print(f"dtype: {dtype}")
print(f"Batch size: {batch_size}")

# Read image
images = [cv2.imread(str(p), cv2.IMREAD_UNCHANGED) for p in image_paths]

Images: 800
Shape: 2800 x 2800
dtype: uint8
Batch size: 40


In [18]:
import hdf5plugin
def write_hdf5_batch(
    output_path,
    batch,
    compression="gzip",
    compression_opts=4,
    chunks=None
):
    batch = np.asarray(batch)
    batch_size, h, w = batch.shape

    kwargs = {}

    if compression == "gzip":
        kwargs["compression"] = "gzip"
        kwargs["compression_opts"] = compression_opts
    elif compression == "lzf":
        kwargs["compression"] = "lzf"
    elif compression == "zstd":
        kwargs.update(hdf5plugin.Zstd(clevel=compression_opts))
    elif compression is not None:
        raise ValueError(f"Unsupported compression: {compression}")

    if chunks is not None:
        kwargs["chunks"] = chunks

    with h5py.File(output_path, "w") as f:
        f.create_dataset(
            "images",
            data=batch,
            **kwargs
        )

        f.attrs["num_images"] = batch_size
        f.attrs["height"] = h
        f.attrs["width"] = w
        f.attrs["dtype"] = str(batch.dtype)
        f.attrs["compression"] = compression if compression else "none"

In [ ]:
png_total_time = 0.0

for start in range(0, n, batch_size):
    end = min(start + batch_size, n)

    batch = images[start:end]

    t0 = time.perf_counter()

    for i, img in enumerate(batch, start=start):
        cv2.imwrite(str(png_out / f"{i:05d}.png"), img)

    png_total_time += time.perf_counter() - t0

print(f"PNG total write time: {png_total_time:.3f} s")
print(f"PNG avg/image: {png_total_time / n * 1000:.3f} ms")
print(f"PNG throughput: {n / png_total_time:.2f} images/s")

PNG total write time: 56.031 s
PNG avg/image: 70.039 ms
PNG throughput: 14.28 images/s


In [ ]:
h5_total_time = 0.0
batch_count = 0

for file in hdf_out.glob("*.h5"):
    file.unlink()

for start in range(0, n, batch_size):
    end = min(start + batch_size, n)
    batch = images[start:end]

    output_path = hdf_out / f"images_{batch_count:04d}.h5"

    t0 = time.perf_counter()

    write_hdf5_batch(
        output_path,
        batch,
        compression="zstd",
        compression_opts=3
    )

    h5_total_time += time.perf_counter() - t0
    batch_count += 1

print(f"HDF5 total write time: {h5_total_time:.3f} s")
print(f"HDF5 avg/image: {h5_total_time / n * 1000:.3f} ms")
print(f"HDF5 throughput: {n / h5_total_time:.2f} images/s")

h5_files = list(hdf_out.glob("*.h5"))
total_size_mb = sum(file.stat().st_size for file in h5_files) / (1024 ** 2)
avg_size_mb = total_size_mb / len(h5_files) if h5_files else 0

print(f"HDF5 total size: {total_size_mb:.2f} MB")
print(f"HDF5 avg/file size: {avg_size_mb:.2f} MB")

HDF5 total write time: 39.294 s
HDF5 avg/image: 49.117 ms
HDF5 throughput: 20.36 images/s
HDF5 total write time: 39.294 s
HDF5 avg/image: 49.117 ms
HDF5 throughput: 20.36 images/s
HDF5 total size: 3493.13 MB
HDF5 avg/file size: 174.66 MB


In [32]:
for cmprs_opt in [3, 5, 7, 9, 11]:
    print(f"Compression_opts: {cmprs_opt}")
    h5_total_time = 0.0
    batch_count = 0

    for file in hdf_out.glob("*.h5"):
        file.unlink()

    for start in range(0, n, batch_size):
        end = min(start + batch_size, n)
        batch = images[start:end]

        output_path = hdf_out / f"images_{batch_count:04d}.h5"

        t0 = time.perf_counter()

        write_hdf5_batch(
            output_path,
            batch,
            compression="zstd",
            compression_opts=cmprs_opt
        )

        h5_total_time += time.perf_counter() - t0
        batch_count += 1

    print(f"    HDF5 total write time: {h5_total_time:.3f} s")
    print(f"    HDF5 avg/image : {h5_total_time / n * 1000:.3f} ms")
    print(f"    HDF5 per file  : {h5_total_time / n * 1000 *40/1000:.3f} per 40 images")
    print(f"    HDF5 throughput: {n / h5_total_time:.2f} images/s")

    h5_files = list(hdf_out.glob("*.h5"))
    total_size_mb = sum(file.stat().st_size for file in h5_files) / (1024 ** 2)
    avg_size_mb = total_size_mb / len(h5_files) if h5_files else 0

    print(f"    HDF5 total size: {total_size_mb:.2f} MB")
    print(f"    HDF5 avg/file size: {avg_size_mb:.2f} MB")

Compression_opts: 3
    HDF5 total write time: 35.944 s
    HDF5 avg/image : 44.931 ms
    HDF5 per file  : 1.797 per 40 images
    HDF5 throughput: 22.26 images/s
    HDF5 total size: 3493.13 MB
    HDF5 avg/file size: 174.66 MB
Compression_opts: 5
    HDF5 total write time: 69.225 s
    HDF5 avg/image : 86.531 ms
    HDF5 per file  : 3.461 per 40 images
    HDF5 throughput: 11.56 images/s
    HDF5 total size: 3331.96 MB
    HDF5 avg/file size: 166.60 MB
Compression_opts: 7
    HDF5 total write time: 115.651 s
    HDF5 avg/image : 144.563 ms
    HDF5 per file  : 5.783 per 40 images
    HDF5 throughput: 6.92 images/s
    HDF5 total size: 3319.68 MB
    HDF5 avg/file size: 165.98 MB
Compression_opts: 9
    HDF5 total write time: 127.380 s
    HDF5 avg/image : 159.226 ms
    HDF5 per file  : 6.369 per 40 images
    HDF5 throughput: 6.28 images/s
    HDF5 total size: 3322.49 MB
    HDF5 avg/file size: 166.12 MB
Compression_opts: 11
    HDF5 total write time: 226.378 s
    HDF5 avg/image :

In [34]:

h5_total_time = 0.0
batch_count = 0

for file in hdf_out.glob("*.h5"):
    file.unlink()

for start in range(0, n, batch_size):
    end = min(start + batch_size, n)
    batch = images[start:end]

    output_path = hdf_out / f"images_{batch_count:04d}.h5"

    t0 = time.perf_counter()

    write_hdf5_batch(
        output_path,
        batch,
        # compression="zstd",
        compression_opts=None
    )

    h5_total_time += time.perf_counter() - t0
    batch_count += 1

print(f"    HDF5 total write time: {h5_total_time:.3f} s")
print(f"    HDF5 avg/image : {h5_total_time / n * 1000:.3f} ms")
print(f"    HDF5 per file  : {h5_total_time / n * 1000 *40/1000:.3f} per 40 images")
print(f"    HDF5 throughput: {n / h5_total_time:.2f} images/s")

h5_files = list(hdf_out.glob("*.h5"))
total_size_mb = sum(file.stat().st_size for file in h5_files) / (1024 ** 2)
avg_size_mb = total_size_mb / len(h5_files) if h5_files else 0

print(f"    HDF5 total size: {total_size_mb:.2f} MB")
print(f"    HDF5 avg/file size: {avg_size_mb:.2f} MB")

    HDF5 total write time: 129.746 s
    HDF5 avg/image : 162.183 ms
    HDF5 per file  : 6.487 per 40 images
    HDF5 throughput: 6.17 images/s
    HDF5 total size: 3339.32 MB
    HDF5 avg/file size: 166.97 MB
